# Problem 012:  Highly Divisible Triangular Number

> The sequence of triangle numbers is generated by adding the natural numbers. So the $7$th triangle number would be $1 + 2 + 3 + 4 + 5 + 6 + 7 = 28$. The first ten terms would be:
>
> $$1, 3, 6, 10, 15, 21, 28, 36, 45, 55, \dots$$
>
> Let us list the factors of the first seven triangle numbers:
>
> $$\begin{align}
\mathbf 1 &\colon 1\\
\mathbf 3 &\colon 1,3\\
\mathbf 6 &\colon 1,2,3,6\\
\mathbf{10} &\colon 1,2,5,10\\
\mathbf{15} &\colon 1,3,5,15\\
\mathbf{21} &\colon 1,3,7,21\\
\mathbf{28} &\colon 1,2,4,7,14,28
\end{align}$$
>
> We can see that $28$ is the first triangle number to have over five divisors.
>
> What is the value of the first triangle number to have over five hundred divisors?


## Sieve algorithm $\mathcal O (???)$

Given the prime factorization of a number $n$ as
$$n = \prod_i p_i^{m_i},$$
The number of divisors of $n$, $\sigma(n)$ is:
$$\sigma(n) = \prod_i (m_i+1).$$
For the case of the triangle numbers, $T_n$, we have:
$$\sigma(T_n) = \sigma\left(\frac{n(n+1)}{2}\right) = \begin{cases} \sigma(n/2)\,\sigma(n+1) & n|2 \\ \sigma(n)\,\sigma((n+1)/2) & n\nmid2 \end{cases}$$
The number of divisors function is multiplicative only when the two values share no prime factors, as is the case of two consecutive numbers.  The piecewise nature of the equation is due to the fact that only one of $n$ and $n+1$ is even.

The code consists of finding the number of divisors for values until the desired target number is reached.  The sieve of Eratosthenes can be modified slightly to make for efficient prime factorization.  Instead of just having a sieve contain `True` and `False` values, the sieve can be updated to contain the least prime factor of the value.  In this case, an element in the sieve is updated only if it is the first time it is updated.  Finding the prime factorization for a number consists of reading the smallest prime divisor off of the sieve, dividing the value by that prime, and continuing the process until the value is at $1$.  Since the sieve I use only contains odd values, initially the factors of $2$ first have to be counted.

As in problem 007, there is no way of knowing _a priori_ how many numbers we will need to factor.  The sieve is thus updated by powers of two until the desired value is found.

I tried to find the asymptotic behaviour for the inverse of the bound of the number of divisors, but was not able to find anything except for the fact that it grows very quickly.  The asymptotics of finding the first triangle number with $N$ divisors has horrible time scaling, but this is probably as good as can be done.  I don't think there would be a way around needing to find all the primes below the index of the triangle number, and the sieve is the way to go.  The slowest part, now, is the prime factorization of a number, which will go as $\mathcal O(\ln n)$ for a single value $n$.  I can imagine a way of storing the number of divisors in the sieve itself, but it would be at the expense of making the sieving process less efficient.


In [11]:
%%time

def count_divisors(x, s):
    ndiv = 1
    while x % 2 == 0:
        ndiv += 1
        x //= 2
    while x != 1:
        p = s[x//2-1]
        m = 2
        x //= p
        while x != 1 and s[x//2-1] == p:
            m += 1
            x //= p
        ndiv *= m
    return ndiv

def extend_sieve(pow2, primes):
    s = [None] * pow2
    # seive previous primes
    for p in primes[1:]:
        if p * p > 4*pow2:
            break
        i0 = ((2 * pow2) // p + 1) * p
        if i0 % 2 == 0:
            i0 += p
        i0 = (i0 - 2*pow2) // 2
        s[i0::p] = [p if a is None else a for a in s[i0::p]]
    # search for new primes
    for i in range(pow2):
        if s[i] is None:
            s[i] = 2*(pow2+i)+1
            primes.append(2*(pow2+i)+1)
    return s
    
def p012(N: int) -> int:
    pow2 = 1
    primes = [2]
    s = []
    i = 2
    nd = 1
    while True:
        if i > 2 * pow2:
            s += extend_sieve(pow2, primes)
            pow2 *= 2
        nd_prev = nd
        if i % 2:
            nd = count_divisors(i, s)
        else:
            nd = count_divisors(i // 2, s)
        if nd * nd_prev > N:
            return (i * (i - 1)) // 2
        i += 1
    return 0

N = 500
ans = p012(N)

print(ans)

76576500
CPU times: user 14.7 ms, sys: 1.99 ms, total: 16.7 ms
Wall time: 16 ms
